# PulsHealth HealthKit database exploration

This notebook connects directly to the PulsHealth Postgres/TimescaleDB database and gives a first-pass look at the normalized HealthKit tables. It intentionally joins through `sample_types.identifier` instead of relying on database-local `type_id` numbers.

Connection settings are loaded from `DATABASE_URL`, `.env`, or `server/.env`. The notebook searches the current working directory and its parents, so it works whether the kernel starts in the repo root or in `notebooks/`. Keep secrets in `.env`; this notebook should not contain the database password.

```bash
python3 -m pip install -r notebooks/requirements.txt
jupyter lab notebooks/healthkit_database_exploration.ipynb
```

Optional settings: `PULS_ANALYSIS_TZ` defaults to `UTC`, `PULS_LOOKBACK_DAYS` defaults to `90`, and `PULS_DB_SAMPLE_LIMIT` defaults to `100`.

In [ ]:
import os
import socket
import subprocess
import time
from pathlib import Path
from urllib.parse import quote_plus

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sqlalchemy as sa
from IPython.display import Markdown, display
from sqlalchemy import text

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 100)

ANALYSIS_TZ = os.getenv("PULS_ANALYSIS_TZ", "UTC")
LOOKBACK_DAYS = int(os.getenv("PULS_LOOKBACK_DAYS", "90"))
SAMPLE_LIMIT = int(os.getenv("PULS_DB_SAMPLE_LIMIT", "100"))

display(Markdown(f"Using timezone `{ANALYSIS_TZ}`, lookback `{LOOKBACK_DAYS}` days, sample limit `{SAMPLE_LIMIT}`."))

In [ ]:
def normalize_database_url(url: str) -> str:
    if url.startswith("postgres://"):
        return "postgresql+psycopg://" + url[len("postgres://"):]
    if url.startswith("postgresql://"):
        return "postgresql+psycopg://" + url[len("postgresql://"):]
    return url


def load_env_file(path: Path) -> dict[str, str]:
    loaded = {}
    if not path.exists():
        return loaded

    for raw_line in path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip('\"').strip("'")
        if key and not os.getenv(key):
            os.environ[key] = value
            loaded[key] = value
    return loaded


def candidate_env_files() -> list[Path]:
    candidates = []
    for base in (Path.cwd(), *Path.cwd().parents):
        candidates.extend((base / ".env", base / "server" / ".env"))
    seen = set()
    unique_candidates = []
    for candidate in candidates:
        resolved = candidate.resolve(strict=False)
        if resolved not in seen:
            seen.add(resolved)
            unique_candidates.append(candidate)
    return unique_candidates


loaded_env_files = []
for env_file in candidate_env_files():
    loaded = load_env_file(env_file)
    if loaded:
        loaded_env_files.append(str(env_file))


def port_is_open(host: str, port: int, timeout: float = 0.4) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def ensure_ssh_tunnel() -> str | None:
    ssh_host = os.getenv("PULS_DB_SSH_HOST")
    if not ssh_host:
        return None

    local_host = os.getenv("PULS_DB_HOST", "127.0.0.1")
    local_port = int(os.getenv("PULS_DB_PORT", "15432"))
    remote_host = os.getenv("PULS_DB_SSH_REMOTE_HOST", "127.0.0.1")
    remote_port = int(os.getenv("PULS_DB_SSH_REMOTE_PORT", "5432"))

    if port_is_open(local_host, local_port):
        return f"existing SSH tunnel via {ssh_host}"

    subprocess.Popen(
        [
            "ssh",
            "-N",
            "-L",
            f"{local_port}:{remote_host}:{remote_port}",
            "-o",
            "BatchMode=yes",
            "-o",
            "ExitOnForwardFailure=yes",
            ssh_host,
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    for _ in range(20):
        if port_is_open(local_host, local_port):
            return f"new SSH tunnel via {ssh_host}"
        time.sleep(0.25)
    raise RuntimeError(f"PULS_DB_SSH_HOST is set to {ssh_host}, but the SSH tunnel did not open.")


def database_url_from_env() -> tuple[str, str]:
    url = os.getenv("DATABASE_URL")
    if url:
        return normalize_database_url(url), "DATABASE_URL"

    tunnel_source = ensure_ssh_tunnel()
    host = os.getenv("PULS_DB_HOST", "127.0.0.1")
    port = os.getenv("PULS_DB_PORT", "5432")
    name = os.getenv("PULS_DB_NAME", "postgres")
    user = os.getenv("PULS_DB_USER", "postgres")
    password = os.getenv("PULS_DB_PASSWORD") or os.getenv("POSTGRES_PASSWORD")
    if not password:
        raise RuntimeError(
            "No database password found. Set DATABASE_URL, or add PULS_DB_PASSWORD "
            "or POSTGRES_PASSWORD to .env/server/.env."
        )

    source = tunnel_source or "env connection settings"
    return f"postgresql+psycopg://{quote_plus(user)}:{quote_plus(password)}@{host}:{port}/{name}", source


DATABASE_URL, CONNECTION_SOURCE = database_url_from_env()
engine = sa.create_engine(DATABASE_URL, pool_pre_ping=True)


def q(query: str, params: dict | None = None) -> pd.DataFrame:
    try:
        return pd.read_sql_query(text(query), engine, params=params or {})
    except sa.exc.SQLAlchemyError as exc:
        engine.dispose()
        return pd.DataFrame({"error": [str(exc)]})


with engine.connect() as conn:
    version = conn.execute(text("select version()")).scalar_one()

source_note = f" Loaded env from `{', '.join(loaded_env_files)}`." if loaded_env_files else ""
display(Markdown(f"Connected to Postgres via `{CONNECTION_SOURCE}`.{source_note}"))
print(version)

In [ ]:
required_relations = [
    "users",
    "sample_types",
    "sources",
    "quantity_samples",
    "category_samples",
    "workouts",
    "batches",
]

relations = q(
    """
    SELECT c.relname AS relation_name, c.relkind
    FROM pg_class c
    JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname = 'public'
      AND c.relkind IN ('r', 'v', 'm')
    ORDER BY c.relname
    """
)
present = set(relations["relation_name"])
missing = sorted(set(required_relations) - present)
if missing:
    raise RuntimeError(f"Database is missing expected PulsHealth relations: {missing}")

row_count_sql = " UNION ALL ".join(
    f"SELECT '{name}'::text AS relation_name, count(*)::bigint AS rows FROM {name}"
    for name in required_relations
)
row_counts = q(row_count_sql).sort_values("rows", ascending=False)
display(row_counts)

In [ ]:
users = q(
    """
    SELECT id, name, email, dob, biological_sex, created_at, updated_at
    FROM users
    ORDER BY updated_at DESC NULLS LAST, created_at DESC
    """
)
display(Markdown("## Users"))
display(users)

sample_type_coverage = q(
    """
    WITH counts AS (
      SELECT type_id, count(*)::bigint AS quantity_rows, 0::bigint AS category_rows
      FROM quantity_samples GROUP BY type_id
      UNION ALL
      SELECT type_id, 0::bigint, count(*)::bigint
      FROM category_samples GROUP BY type_id
    )
    SELECT st.identifier, st.kind, st.unit,
           sum(quantity_rows)::bigint AS quantity_rows,
           sum(category_rows)::bigint AS category_rows,
           (sum(quantity_rows) + sum(category_rows))::bigint AS total_rows
    FROM sample_types st
    LEFT JOIN counts c USING (type_id)
    GROUP BY st.identifier, st.kind, st.unit
    ORDER BY total_rows DESC NULLS LAST, st.identifier
    """
)
display(Markdown("## HealthKit type coverage"))
display(sample_type_coverage.head(50))

In [ ]:
recent_batches = q(
    """
    SELECT received_at, trigger, reason, type_identifier,
           sample_count, deletion_count,
           COALESCE(aggregate_count, 0) AS aggregate_count,
           COALESCE(activity_summary_count, 0) AS activity_summary_count,
           bytes, parse_ms, insert_ms
    FROM batches
    ORDER BY received_at DESC
    LIMIT :limit
    """,
    {"limit": SAMPLE_LIMIT},
)
display(Markdown("## Recent upload batches"))
display(recent_batches)

batch_daily = q(
    """
    SELECT (received_at AT TIME ZONE :tz)::date AS day,
           COALESCE(trigger, 'unknown') AS trigger,
           count(*)::bigint AS batches,
           sum(COALESCE(sample_count, 0))::bigint AS samples,
           sum(COALESCE(bytes, 0))::bigint AS bytes
    FROM batches
    WHERE received_at >= now() - (:lookback_days * interval '1 day')
    GROUP BY day, trigger
    ORDER BY day DESC, trigger
    """,
    {"tz": ANALYSIS_TZ, "lookback_days": LOOKBACK_DAYS},
)
display(Markdown("## Upload batches by local day and trigger"))
display(batch_daily)

In [ ]:
quantity_overview = q(
    """
    SELECT st.identifier, st.unit,
           count(*)::bigint AS rows,
           min(qs.start_ts) AS first_sample,
           max(qs.start_ts) AS last_sample,
           count(DISTINCT qs.source_id)::bigint AS sources
    FROM quantity_samples qs
    JOIN sample_types st USING (type_id)
    GROUP BY st.identifier, st.unit
    ORDER BY rows DESC, st.identifier
    """
)
display(Markdown("## Quantity sample overview"))
display(quantity_overview.head(50))

if not quantity_overview.empty:
    plot_data = quantity_overview.head(20).sort_values("rows")
    ax = plot_data.plot.barh(x="identifier", y="rows", figsize=(10, 7), legend=False)
    ax.set_title("Top quantity types by raw row count")
    ax.set_xlabel("Rows")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

In [ ]:
BODY_MASS_IDENTIFIER = "HKQuantityTypeIdentifierBodyMass"

body_mass = q(
    """
    SELECT qs.start_ts,
           qs.value AS kg,
           qs.value * 2.20462262185 AS lb,
           s.name AS source_name,
           s.bundle_id,
           qs.metadata
    FROM quantity_samples qs
    JOIN sample_types st USING (type_id)
    LEFT JOIN sources s USING (source_id)
    WHERE st.identifier = :identifier
    ORDER BY qs.start_ts DESC
    LIMIT :limit
    """,
    {"identifier": BODY_MASS_IDENTIFIER, "limit": SAMPLE_LIMIT},
)
display(Markdown("## Body mass samples"))
display(body_mass)

if not body_mass.empty:
    plot_data = body_mass.sort_values("start_ts")
    ax = plot_data.plot(x="start_ts", y="lb", marker="o", figsize=(10, 4), legend=False)
    ax.set_title("Body mass")
    ax.set_xlabel("Time")
    ax.set_ylabel("lb")
    plt.tight_layout()
    plt.show()
else:
    display(Markdown(f"No rows found for `{BODY_MASS_IDENTIFIER}`."))

In [ ]:
HEART_RATE_IDENTIFIER = "HKQuantityTypeIdentifierHeartRate"

heart_rate_daily = q(
    """
    SELECT (qs.start_ts AT TIME ZONE :tz)::date AS day,
           count(*)::bigint AS rows,
           min(qs.value) AS min_bpm,
           avg(qs.value) AS avg_bpm,
           max(qs.value) AS max_bpm
    FROM quantity_samples qs
    JOIN sample_types st USING (type_id)
    WHERE st.identifier = :identifier
      AND qs.start_ts >= now() - (:lookback_days * interval '1 day')
    GROUP BY day
    ORDER BY day
    """,
    {"tz": ANALYSIS_TZ, "identifier": HEART_RATE_IDENTIFIER, "lookback_days": LOOKBACK_DAYS},
)
display(Markdown("## Daily heart-rate summary"))
display(heart_rate_daily)

if not heart_rate_daily.empty:
    ax = heart_rate_daily.plot(x="day", y=["min_bpm", "avg_bpm", "max_bpm"], figsize=(10, 4))
    ax.set_title("Heart rate by local day")
    ax.set_xlabel("Day")
    ax.set_ylabel("bpm")
    plt.tight_layout()
    plt.show()

In [ ]:
category_samples = q(
    """
    SELECT cs.start_ts, cs.end_ts, st.identifier, cs.value,
           cl.label, cl.enum_name,
           s.name AS source_name, cs.metadata
    FROM category_samples cs
    JOIN sample_types st USING (type_id)
    LEFT JOIN category_labels cl
      ON cl.type_identifier = st.identifier
     AND cl.value = cs.value
    LEFT JOIN sources s USING (source_id)
    ORDER BY cs.start_ts DESC
    LIMIT :limit
    """,
    {"limit": SAMPLE_LIMIT},
)
display(Markdown("## Category samples with HealthKit labels"))
display(category_samples)

In [ ]:
workouts = q(
    """
    SELECT w.start_ts, w.end_ts, w.activity_type,
           w.duration_s / 60.0 AS duration_min,
           w.energy_kcal, w.distance_m,
           s.name AS source_name,
           w.uuid
    FROM workouts w
    LEFT JOIN sources s USING (source_id)
    ORDER BY w.start_ts DESC
    LIMIT :limit
    """,
    {"limit": SAMPLE_LIMIT},
)
display(Markdown("## Recent workouts"))
display(workouts)

In [ ]:
metric_daily_exists = "metric_daily" in set(relations["relation_name"])
if metric_daily_exists:
    metric_daily = q(
        """
        SELECT day, identifier, value, source
        FROM metric_daily
        WHERE day >= (current_date - (:lookback_days * interval '1 day'))::date
        ORDER BY day DESC, identifier
        LIMIT :limit
        """,
        {"lookback_days": LOOKBACK_DAYS, "limit": SAMPLE_LIMIT},
    )
    display(Markdown("## Daily metric view"))
    display(metric_daily)
else:
    display(Markdown("`metric_daily` is not present in this database."))

## Custom query starting points

Use the `q()` helper for ad-hoc analysis. Keep filtering by HealthKit identifier through `sample_types`:

```python
q("""
SELECT qs.start_ts, qs.value, st.unit, s.name AS source_name
FROM quantity_samples qs
JOIN sample_types st USING (type_id)
LEFT JOIN sources s USING (source_id)
WHERE st.identifier = :identifier
ORDER BY qs.start_ts DESC
LIMIT 100
""", {"identifier": "HKQuantityTypeIdentifierRestingHeartRate"})
```

For category values, join `category_labels` using `(sample_types.identifier, category_samples.value)`.

In [ ]:
q("""
SELECT a.*, s.identifier, s.unit
FROM aggregate_series a
JOIN sample_types s
    ON a.type_id = s.type_id

""")

In [ ]:
df = q("""
SELECT bucket_start, bucket_end, value
FROM aggregate_samples
WHERE series_id = 755
    AND value is not null
ORDER BY bucket_start
"""
)

In [ ]:
df[df.value > 250].value.hist(bins=100)

In [ ]:
plt.plot(df.bucket_start, df.value)

In [ ]:
df.sort_values(by='value', ascending=False)

In [ ]:
workouts = q("""
SELECT *
from workouts
"""
)

In [ ]:
routes = q("""
SELECT r.*
FROM workout_route_points r
JOIN workouts w
    ON r.workout_uuid = w.uuid
WHERE w.activity_type = 'running'
    AND w.start_ts >= '2026-01-01'
"""
)

In [ ]:
routes